In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine("postgresql+psycopg2://jeffrey@localhost:5433/marketing")

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).fetchone()[0])

PostgreSQL 16.14 (Postgres.app) on aarch64-apple-darwin21.6.0, compiled by Apple clang version 14.0.0 (clang-1400.0.29.102), 64-bit


In [4]:
df = pd.read_csv("data/marketing_segmented.csv")
df.to_sql("customers", engine, if_exists="replace", index=False)
print("Loaded", len(df), "rows into the 'customers' table ✅")

Loaded 56000 rows into the 'customers' table ✅


In [5]:
pd.read_sql("SELECT COUNT(*) AS total_customers FROM customers", engine)

,total_customers
0,56000


In [8]:
pd.read_sql("""
    SELECT id, income, total_spend, response
    FROM customers
    WHERE income > 75000
    LIMIT 5
""", engine)

,id,income,total_spend,response
0,13664263,98584.6,1512,0
1,2330269,111080.3,843,0
2,923176,104661.2,2109,0
3,9974730,105437.3,1415,1
4,2275327,103586.0,1567,1


In [9]:
pd.read_sql("""
    SELECT income_band,
           COUNT(*) AS customers,
           ROUND(AVG(response) * 100, 1) AS response_rate_pct
    FROM customers
    GROUP BY income_band
    ORDER BY response_rate_pct DESC
""", engine)

,income_band,customers,response_rate_pct
0,100k+,7277,27.8
1,75-100k,12505,21.0
2,50-75k,12575,15.0
3,25-50k,10966,10.4
4,<25k,12677,4.6


In [10]:
pd.read_sql("""
    SELECT
        CASE WHEN children = 0 THEN 'No children' ELSE 'Has children' END AS family_status,
        COUNT(*) AS customers,
        ROUND(AVG(response) * 100, 1) AS response_rate_pct,
        ROUND(AVG(total_spend)) AS avg_spend
    FROM customers
    GROUP BY family_status
""", engine)

,family_status,customers,response_rate_pct,avg_spend
0,Has children,38222,11.6,500.0
1,No children,17778,21.5,943.0


In [11]:
pd.read_sql("""
    SELECT id, income_band, total_spend,
           ROUND(AVG(total_spend) OVER (PARTITION BY income_band)) AS band_avg_spend
    FROM customers
    LIMIT 10
""", engine)

,id,income_band,total_spend,band_avg_spend
0,10426620,100k+,644,1228.0
1,10870763,100k+,355,1228.0
2,15859461,100k+,701,1228.0
3,4408298,100k+,1301,1228.0
4,9421858,100k+,319,1228.0
5,9499406,100k+,816,1228.0
6,16443461,100k+,2388,1228.0
7,5814258,100k+,2971,1228.0
8,14341078,100k+,1905,1228.0
9,10219372,100k+,752,1228.0


In [12]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW v_response_by_income AS
        SELECT income_band,
               COUNT(*) AS customers,
               ROUND(AVG(response) * 100, 1) AS response_rate_pct
        FROM customers
        GROUP BY income_band
    """))
print("View 'v_response_by_income' created ✅")

View 'v_response_by_income' created ✅


In [13]:
pd.read_sql("SELECT * FROM v_response_by_income", engine)

,income_band,customers,response_rate_pct
0,75-100k,12505,21.0
1,100k+,7277,27.8
2,25-50k,10966,10.4
3,<25k,12677,4.6
4,50-75k,12575,15.0


In [14]:
import os
from pandas.io.sql import get_schema

os.makedirs("sql", exist_ok=True)

# 01_schema.sql — DDL (auto-generate the CREATE TABLE from your data)
with open("sql/01_schema.sql", "w") as f:
    f.write("-- DDL: create the customers table\n")
    f.write(get_schema(df, "customers", con=engine) + ";\n")

# 03_queries.sql — analytical queries
with open("sql/03_queries.sql", "w") as f:
    f.write("""-- Analytical queries

-- Response rate by income band
SELECT income_band, COUNT(*) AS customers,
       ROUND(AVG(response)*100, 1) AS response_rate_pct
FROM customers GROUP BY income_band ORDER BY response_rate_pct DESC;

-- Response & spend by family status (CASE)
SELECT CASE WHEN children = 0 THEN 'No children' ELSE 'Has children' END AS family_status,
       COUNT(*) AS customers,
       ROUND(AVG(response)*100, 1) AS response_rate_pct,
       ROUND(AVG(total_spend)) AS avg_spend
FROM customers GROUP BY family_status;

-- Spend vs income-band average (window function)
SELECT id, income_band, total_spend,
       ROUND(AVG(total_spend) OVER (PARTITION BY income_band)) AS band_avg_spend
FROM customers;
""")

# 04_views.sql — dashboard view
with open("sql/04_views.sql", "w") as f:
    f.write("""-- Views for the dashboard
CREATE OR REPLACE VIEW v_response_by_income AS
SELECT income_band, COUNT(*) AS customers,
       ROUND(AVG(response)*100, 1) AS response_rate_pct
FROM customers GROUP BY income_band;
""")

print("SQL files written ✅:", os.listdir("sql"))

SQL files written ✅: ['04_views.sql', '03_queries.sql', '01_schema.sql']
